In [ ]:
最適なペア順に信号線値を並べ替えたファイル（s~simular_output）をもとに、最適なペアの信号線を統合し、正解データを作成
※s~simular_outputファイルは、s~outputファイルの行と列が入れ替わっていることに注意⇒詳しくはokinaka.txtを参照

In [37]:
# 学習済みのANNを用いて学習を行うプログラム
# cd workspace/research2/experiment
#　実行コマンド：　python3 diagnosis.py

import numpy as np
import logging
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, model_from_json
from keras.layers import Dense, Dropout, BatchNormalization, Activation
from keras.layers import LeakyReLU
from tensorflow.keras.utils import get_custom_objects
from keras import optimizers
from keras import backend as K
from tensorflow.keras.optimizers import Adam

import matplotlib
import matplotlib.pyplot as plt
from memory_profiler import memory_usage
import time
import pathlib
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

import sys
import argparse
import os
import multiprocessing
from multiprocessing import Process

import random

#グローバル変数
cir = 's344'  # 対象回路
tp_file = cir + '.vec'  # テストパターンファイル名
part_stdic_file = cir + "stdic_bi/aout_" #縮退故障辞書ファイル名の一部
part_brdic_file = cir + "brdic_bi/aout_" #ブリッジ故障辞書ファイルの一部
diagnosis_dir = cir + "diagnosis_data/" #故障診断を行うためのデータを保存するフォルダ
target_fault_num = 30  # 故障診断対象の故障の数⇒全ての故障の中からtarget_fault_num個をランダムに選択する
correct_output_file = 'correct_output/' + cir + '_correct_output'  # 正常な回路の出力ファイル
input_data_file = cir + 'input'  # 入力データファイル
correct_data_file = cir + '分割正解データ' + '/' + cir + 'integrated_output'  # 統合後の正解データファイル
suplit_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_num'  # モデルの分割数が保存されたファイル
suplit_data_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_data_num'  # データの分割数が保存されたファイル
input_data_num = None  # 1個のモデルにおける学習データ数
input_node_num = None  #入力層におけるノード数 初期値は8　学習結果によって変更
output_node_num = None #　出力層のノード数＝分割数による
num_models = None  # 学習させるモデルの数
model_folder = cir + 'sepmodel'  # 学習済みモデルを保存するフォルダ
fault_line_num = None  # 故障診断対象の回路における故障信号線の総数
fault_type_sum = 12  # 故障の種類の総数

processes = 8  # 並列処理のプロセス数

def mk_input_data():
    # 入力データファイルを開いてデータを読み込む
    with open(input_data_file) as f:
        lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]

    # print(lines)
    
    global input_data_num
    input_data_num = int(len(lines))      #学習データ数を設定。学習データ数は入力データの行数
    global input_node_num               #グローバル変数を書き換え
    input_node_num = int(len(lines[0]))      #入力ノード数を設定。入力ノード数は入力データの各行の要素数

    int_lines = [list(map(int, _)) for _ in lines]  #list要素の型をint型に変換

    return np.array(int_lines)


def mk_output_data(fname):
    # 正解データファイルを開いてデータを読み込む
    with open(fname) as f:
        lines = [_.replace("\n", "") for _ in f.readlines()]
    
    lines = [value.split(",") for value in lines] # カンマ区切りで各信号線をリストに格納
    
    # print("dsadsa")
    # print(lines[0])

    for i in range(len(lines)):
        for j in range(len(lines[i])):
            lines[i][j] = float(lines[i][j])
    
    # print("gaga")
    # print(lines[0])
    
    global output_node_num               #グローバル変数を書き換え
    output_node_num = int(len(lines[0]))      #出力ノード数を設定。出力ノード数は正解データの各行の要素数


if __name__ == '__main__':
    
    # テストパターン数を取得
    with open(tp_file, 'r') as f:
        line = f.readline()
        tp_num = int(line.split()[0])  # テストパターン数
        print(tp_num)

    # 故障診断対象の回路における故障の総数を取得
    part_stdic_file0 =  part_stdic_file + str(0)  # 故障辞書ファイルの1つ
    with open(part_stdic_file0, 'r') as f:
        fault_inf = f.readline()  # 故障情報
        fault_num = int(fault_inf.split()[2])  # 対象回路で起こりうる故障数
        lines = f.readlines()[::2]  # 故障辞書ファイルの全行を読み込む
        st_fault_all_line = [int(line.split()[3]) for line in lines[::2]]  # 故障信号線のIDを取得
        print(fault_num)
        print("st_fault_all_line：", st_fault_all_line)
        # print(suplit_num)

    # 縮退故障が発生する信号線をtarget_fault_num個ランダムに選択。さらに、その信号線で発生するのは0縮退故障か1縮退故障かをランダムに決定する
    st_fault_target_line = random.sample(st_fault_all_line, target_fault_num) # 想定する故障信号線をランダムに選ぶ
    st_fault_target_line = sorted(st_fault_target_line) # 小さい番号順に並び替える
    st_fault_target_type = random.choices([0, 1], k=target_fault_num)  # 0縮退故障か1縮退故障かをランダムに決定する
    with open(diagnosis_dir + 'st_fault_line', 'w') as f: # 故障IDをファイルに保存
        f.write('診断対象縮退故障数' + str(target_fault_num) + '\n')  # 故障の数をファイルに保存
        for i in range(target_fault_num):
            f.write(str(st_fault_target_line[i]) + ' ' + "sa " + str(st_fault_target_type[i]) + "\n")

    print(st_fault_target_line)
    print(st_fault_target_type)

    # 診断対象故障の出力値をファイルに保存
    for i in range(tp_num):
      stdic_file = part_stdic_file + str(i)  # 故障辞書ファイルを指定
      with open(stdic_file, 'r') as f:
        fault_inf = f.readline()  # 故障情報
        lines = f.readlines()  # 故障辞書ファイルの全行を読み込む

      # 各辞書におけるIDと出力値を取得
      fault_output = [0 for _ in range(target_fault_num)]  # 故障出力値を格納するリスト
      idx = 0
      for j in range(0, len(lines), 2):
        if st_fault_target_line[idx] == int(lines[j].split()[3]) and st_fault_target_type[idx] == int(lines[j].split()[5]): # ファイルの「id 0 Fault 1 sa 0」の行から、信号線番号と0、1縮退故障情報を取得して、比較
            fault_output[idx] = lines[j+1].replace('\n', '')  # 故障出力値を取得
            idx += 1
        if idx == target_fault_num:  # 診断対象の故障数に達したら、ループを抜ける。これは、診断対象の故障信号線が小さい順に並び替えられているため、target_fault_num個の故障を取得したら、ループを抜ける
            break

      # 取得した故障出力値をファイルに保存
      with open(diagnosis_dir + 'fault_output' + str(i), 'w') as f:
          for j in range(len(fault_output)):
              f.write(fault_output[j] + '\n')
    
    print(f"fault_output：{fault_output}")
    
    #故障出力を正常な回路出力と比較して、パスフェイル情報を取得する
    correct_output_value = []  # 正常な回路出力を格納するリスト
    with open(correct_output_file, 'r') as f:
        lines = f.readlines()  # 正常な回路出力の全行を読み込む
        for i in range(1, len(lines), 2):
            correct_output_value.append(lines[i].replace('\n', ''))

    # print(correct_output_value)
    
    # 各故障の種類における回路出力と正常な回路出力を比較して、パスフェイル情報を取得
    fault_output_value = [[0 for _ in range(tp_num)] for _ in range(target_fault_num)]  # 故障出力値を格納する二次元リスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される
    for i in range(tp_num):
        with open(diagnosis_dir + 'fault_output' + str(i), 'r') as f:
            lines = f.readlines()  # fault_output~ファイルから故障出力値の全行を読み込む
            for j in range(target_fault_num):
                fault_output_value[j][i] = lines[j].replace('\n', '')
    
    print(f"fault_output_value{fault_output_value}")
    
    # 故障出力値と正常な回路出力を比較して、パスフェイル情報を取得
    pass_fail = [[0 for _ in range(tp_num)] for _ in range(target_fault_num)]  # パスフェイル情報を格納するリスト。診断を行う際に使用する
    for i in range(target_fault_num):
        with open(diagnosis_dir + 'pass_fail' + str(i), 'w') as f:
            for j in range(tp_num):
                if correct_output_value[j] == fault_output_value[i][j]:
                    pass_fail_value = '0' # pass 正常な回路出力と一致
                    pass_fail[i][j] = 0
                else:
                    pass_fail_value = '1' # fail 故障が発生した回路の出力値と正常な回路出力が一致しない
                    pass_fail[i][j] = 1
                
                f.write(pass_fail_value)

    print(f"pass_fail{pass_fail}")
    
  # 学習済みの機械学習モデルに入力データを与えて、出力を取得する
    # 入力データをファイルから読み込む
    input_data = mk_input_data()

    # 分割モデルの数を取得
    with open(suplit_num_file, 'r') as f:
        num_models = int(f.readline())
    
    # 診断対象の回路における故障信号線の総数を取得
    with open(cir + 'output', 'r') as f:
        line = f.readline().replace(",", "").replace("\n", "")
        global faulet_line_num
        fault_line_num = int(len(line))  # 診断対象の故障信号線の総数
        print("fault_line_num" + str(fault_line_num))

    # 故障の種類ごとに、パスフェイル情報を格納するリストを作成
    model_pass_fail = [[0 for _ in range(fault_line_num)] for _ in range(input_data_num)]  # 故障出力値を格納するリスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される


    with open(cir + 'pair_list', 'r') as f:
        integration_num = int(f.readline().split()[1])  # 統合数
        # print(integration_num)
        signal_num = int(f.readline().split()[1])  # 信号線の数
        print("signal_num", signal_num)
        lines = [line.split() for line in f.readlines()]  # 故障信号線のペアを取得
        print(lines)
    
    # 信号線ペアのIDをリストに格納
    print("line_id:" + str(integration_num * len(lines)))
    line_id = [0 for _ in range(signal_num - (signal_num%integration_num))]  # 故障信号線のIDを格納するリスト
    for i in range(len(lines) - (signal_num%integration_num)):    # signal_num%integration_numは、信号線が奇数かどうか(integration)を考慮。
        for j in range(integration_num):
            line_id[i*integration_num + j] = int(lines[i][j])  # 故障信号線のIDを取得
    
    # if signal_num % integration_num != 0:  # 信号線の数が奇数の場合、ペアができていない信号線のIDをリストに追加
    #     for i in range(signal_num % integration_num):
    #         line_id[len(lines) - (signal_num % integration_num) + i] = int(lines[len(lines) - (signal_num % integration_num)][i])
    
    print(line_id)
    print(len(line_id))

    # データを何個づつ分割したのかを取得
    with open(suplit_data_num_file, 'r') as f:
        suplit_data_num = int(f.readline().replace("\n", ""))  # データの分割数を取得
        print(suplit_data_num)

    print(f"input_data_num:{input_data_num}")

    a = [0 for _ in range(input_data_num)]  # input_data_num個の要素を持つリストを作成

    for model_id in range(num_models):

        # モデルの読み込み
        with open(model_folder + '/' + cir + 'model_' + str(model_id) + '.tflite', 'rb') as f:  # モデルを読み込むファイルを開く
            tflite_model = f.read()

        # モデルの評価
        interpreter = tf.lite.Interpreter(model_content=tflite_model)  # TFLite形式のモデルを読み込む。保存されたTFLiteモデルをメモリに読み込み、推論を行う準備をするためのインタープリターを作成します。
        interpreter.allocate_tensors()  # #メモリを確保。モデルが使用するテンソル（データ構造）をメモリに割り当てます。TFLiteモデルをインタープリターにロードするだけでは、テンソルのメモリは割り当てられていません。この行を実行することで、モデルが推論に必要なメモリを確保します。

        input_details = interpreter.get_input_details()  # モデルの入力テンソルの詳細を取得。モデルの入力に関する詳細情報をリスト形式で返します。各要素は、入力テンソルの形状、データ型、名前などの情報を含む辞書です。
        output_details = interpreter.get_output_details()  # モデルの出力テンソルの詳細を取得。モデルの出力に関する詳細情報をリスト形式で返します。各要素は、出力テンソルの形状、データ型、名前などの情報を含む辞書です。
        
        n = 0
        for i in range(input_data_num):
            
            input_shape = input_details[0]['shape']  # 入力データの形状を取得.先ほど取得したinput_detailsのリストの中から、形状情報を取得。input_details[0]['shape']は、入力テンソルの形状を取得するためのコードです。['shape']は、辞書内の shape キーを指定しています。

            reshape_input_data = np.reshape(input_data[i], input_shape)  # NumPy配列の形状を指定した形 (input_shape) に変更します。これにより、入力データの形状がモデルの入力テンソルの形状と一致します。


            # モデルの入力データを設定
            interpreter.set_tensor(input_details[0]['index'], np.array(reshape_input_data, dtype=np.float32))   # モデルの入力テンソルにデータを設定します。入力テンソルのインデックス、データを指定します。[0]['index']は、入力テンソルのインデックスを取得するためのコードです。

            # モデルの推論を実行
            interpreter.invoke()

            # モデルの出力データを取得
            output_data = interpreter.get_tensor(output_details[0]['index'])

            correct_file = correct_data_file + str(model_id)  # 分割された正解データファイル名に変更　＝　s344分割正解データ/s344integrated_output + 番号
            mk_output_data(correct_file)  # 関数内でグローバル変数output_node_numを設定

            for j in range(output_node_num):
                if output_data[0][j] < 0.25:
                    output_data[0][j] = 0
                elif output_data[0][j] < 0.5:
                    output_data[0][j] = 0.375
                elif output_data[0][j] < 0.75:
                    output_data[0][j] = 0.625
                else:
                    output_data[0][j] = 1
            
            count = 0  # 統合したものの統合を解くために必要なカウント変数
            if model_id != (num_models - 1):  # 最後のモデル以外の場合
                for j in range(output_node_num):
                    # print(j + count + integration_num*suplit_data_num*model_id)
                    # print(f"line_id: {line_id[j + count + integration_num*suplit_data_num*model_id]}")
                    if output_data[0][j] == 0:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.375:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.625:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    else:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    # if line_id[j + count + integration_num*suplit_data_num*model_id] == 0:
                    #     a[n] = model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]
                    #     n += 1
                    count += 1
            else:  # 最後のモデルの場合
                for j in range(output_node_num - 1):
                    # print("だｋｆｄｊｊｈだｌｋｈふぃｄぱふぁｄｆだｋｆｄｊｊｈだｌｋｈふぃｄぱふぁｄｆ")
                    # print(j + count + integration_num*suplit_data_num*model_id)
                    # print(f"line_id: {line_id[j + count + integration_num*suplit_data_num*model_id]}")
                    if output_data[0][j] == 0:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.375:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 0
                    elif output_data[0][j] == 0.625:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 0
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    else:
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]= 1
                        model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id + 1]] = 1
                    # if line_id[j + count + integration_num*suplit_data_num*model_id] == 183:
                    #     a[n] = model_pass_fail[i][line_id[j + count + integration_num*suplit_data_num*model_id]]
                    #     n += 1
                    count += 1


        
            # print(output_data)                # output_dataには、テストパターンnのときの、故障があるかどうかの情報が格納されている

    # print(model_pass_fail[0])
    # print(model_pass_fail[1])
    # print(model_pass_fail[2])
    # print(model_pass_fail[3])
    # print(a)


  # モデル出力を故障の種類（fault_type_sum個）に分ける
    # model_pass_failは、行はモデルの入力データ数、列は故障信号線の数を表す二次元リストであるため、パスフェイル情報（pass_failnファイル）と比較するために、行と列を入れ替える
    model_pass_fail = [[row[i] for row in model_pass_fail] for i in range(len(model_pass_fail[0]))]  #内包表記を使って行と列を入れ替える

    compare_model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(signal_num*tp_num)] # 各行が各故障が起きた場合の各テストパターンのパスフェイル情報を格納する二次元リスト
    
    for i in range(signal_num):
        for j in range(fault_type_sum):
            for k in range(tp_num):
                compare_model_pass_fail[i*fault_type_sum + j][k] = model_pass_fail[i][k*fault_type_sum + j] 

    print("compare_model_pass_fail", compare_model_pass_fail)


    
    # ランダムに選んだ故障のパスフェイル情報（pass_failリスト）とcompare_model_pass_failを比較して、故障候補を取得する
    fault_candidate = [[] for _ in range(target_fault_num)] # 故障候補を格納するリスト
    fault_type = [[] for _ in range(target_fault_num)] # 故障の種類を格納するリスト 0=0縮退故障、1=1縮退故障、2=ブリッジ故障➀、3=ブリッジ故障➁、4=ブリッジ故障➂、5=ブリッジ故障➃、6=ブリッジ故障➄、7=ブリッジ故障➅、8=ブリッジ故障➆、9=ブリッジ故障➇、10=ブリッジ故障➈、11=ブリッジ故障➉
    print(len(st_fault_all_line))
    for i in range(target_fault_num):
        line_count = 0
        count = 0
        for j in range(signal_num*fault_type_sum):
            if compare_model_pass_fail[j] == pass_fail[i]:  # パスフェイル情報を比較
                print(line_count, len(fault_candidate[i]))
                fault_candidate[i].append(st_fault_all_line[line_count])  # 故障候補を格納するリストに追加
                fault_type[i].append(count%fault_type_sum)  # 故障の種類を格納するリストに追加
            
            if count == (fault_type_sum - 1):  # compare_model_pass_failは、12行ごとが信号線に対応しているため、12行ごとにカウントをリセットする
                line_count += 1
                count = 0
            
            count += 1
    
    # 故障候補の中に、実際に故障として選んだ信号線と故障の種類が一致するものがあるかどうかを確認
    for i in range(target_fault_num):
        for j in range(len(fault_candidate[i])):
            if fault_candidate[i][j] == st_fault_target_line[i] and fault_type[i][j] == st_fault_target_type[i]:
                print("故障候補の中に実際に選んだ故障 ", fault_candidate[i][j], " の ", fault_type[i][j], " 縮退故障が含まれていました")
                with open(diagnosis_dir + 'fault_candidate', 'a') as f:
                    f.write(str(fault_candidate[i][j]) + ' ' + str(fault_type[i][j]) + '\n')
                break
            if j == len(fault_candidate[i]) - 1:
                print("故障候補の中に実際に選んだ故障 ", fault_candidate[i][j], " の ", fault_type[i][j], " 縮退故障は含まれていませんでした")
                with open(diagnosis_dir + 'fault_candidate', 'a') as f:
                    f.write(str(fault_candidate[i][j]) + ' ' + str(fault_type[i][j]) + '\n')



    # それぞれの故障
    # model_pass_fail_sa0 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # 0縮退故障出力値を格納するリスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される
    # model_pass_fail_sa1 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # 1縮退故障
    # model_pass_fail_br0 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➀
    # model_pass_fail_br1 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➁
    # model_pass_fail_br2 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➂
    # model_pass_fail_br3 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➃ 
    # model_pass_fail_br4 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➄
    # model_pass_fail_br5 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➅
    # model_pass_fail_br6 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➆
    # model_pass_fail_br7 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➇
    # model_pass_fail_br8 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➈
    # model_pass_fail_br9 = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # ブリッジ故障➉


    


  
    




22
670
st_fault_all_line： [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 

IndexError: list index out of range

In [33]:
# 例: target_fault_num = 3 の場合
fault_candidate = [[], [], []]

# 1つ目の故障候補リストに値を追加
fault_candidate[0].append(100)  # [[100], [], []]
fault_candidate[0].append(200)  # [[100], [200], []]
print(fault_candidate)  # [[100], [], []]
print(fault_candidate[0].append(200))  # [[100], [200], []]
print(st_fault_all_line[334])

[[100, 200], [], []]
None
361


In [5]:
# 学習済みのANNを用いて故障診断を行うプログラム
# cd workspace/research2/experiment
#　実行コマンド：　python3 diagnosis.py

import numpy as np
import tensorflow as tf

import random

#グローバル変数
cir = 's344'  # 対象回路
tp_file = cir + '.vec'  # テストパターンファイル名
part_stdic_file = cir + "stdic_bi/aout_" #縮退故障辞書ファイル名の一部
part_brdic_file = cir + "brdic_bi/aout_" #ブリッジ故障辞書ファイルの一部
st_diagnosis_dir = cir + "diagnosis_st_data/" #縮退故障診断を行うためのデータを保存するフォルダ
br_diagnosis_dir = cir + "diagnosis_br_data/" #ブリッジ故障診断を行うためのデータを保存するフォルダ
target_fault_num = 30  # 故障診断対象の故障の数⇒全ての故障の中からtarget_fault_num個をランダムに選択する
correct_output_file = 'correct_output/' + cir + '_correct_output'  # 正常な回路の出力ファイル
input_data_file = cir + 'input'  # 入力データファイル
correct_data_file = cir + '分割正解データ' + '/' + cir + 'integrated_output'  # 統合後の正解データファイル
suplit_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_num'  # モデルの分割数が保存されたファイル
suplit_data_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_data_num'  # データの分割数が保存されたファイル
single_line_file = cir + '分割正解データ' + '/' + cir + 'single_line'  # 統合されていない信号線があるかが保存されたファイル
input_data_num = None  # 1個のモデルにおける学習データ数
input_node_num = None  #入力層におけるノード数 初期値は8　学習結果によって変更
output_node_num = None #　出力層のノード数＝分割数による
num_models = None  # 学習させるモデルの数
model_folder = cir + 'sepmodel'  # 学習済みモデルを保存するフォルダ
fault_line_num = None  # 故障診断対象の回路における故障信号線の総数
fault_type_sum = 12  # 故障の種類の総数

processes = 8  # 並列処理のプロセス数

def mk_input_data():
    # 入力データファイルを開いてデータを読み込む
    with open(input_data_file) as f:
        lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]

    # print(lines)
    
    global input_data_num
    input_data_num = int(len(lines))      #学習データ数を設定。学習データ数は入力データの行数
    global input_node_num               #グローバル変数を書き換え
    input_node_num = int(len(lines[0]))      #入力ノード数を設定。入力ノード数は入力データの各行の要素数

    int_lines = [list(map(int, _)) for _ in lines]  #list要素の型をint型に変換

    return np.array(int_lines)


def mk_output_data(fname):
    # 正解データファイルを開いてデータを読み込む
    with open(fname) as f:
        lines = [_.replace("\n", "") for _ in f.readlines()]
    
    lines = [value.split(",") for value in lines] # カンマ区切りで各信号線をリストに格納
    
    # print("dsadsa")
    # print(lines[0])

    for i in range(len(lines)):
        for j in range(len(lines[i])):
            lines[i][j] = float(lines[i][j])
    
    # print("gaga")
    # print(lines[0])
    
    global output_node_num               #グローバル変数を書き換え
    output_node_num = int(len(lines[0]))      #出力ノード数を設定。出力ノード数は正解データの各行の要素数
    print("output_node_num:", output_node_num)




if __name__ == '__main__':
    
    # テストパターン数を取得
    with open(tp_file, 'r') as f:
        line = f.readline()
        tp_num = int(line.split()[0])  # テストパターン数
        print(tp_num)

    # 故障診断対象の回路における「縮退故障」の総数を取得
    with open(part_stdic_file + str(0), 'r') as f:  # 縮退故障辞書ファイルの1つを開く
        fault_inf = f.readline()  # 故障情報
        st_fault_num = int(fault_inf.split()[2])  # 対象回路で起こりうる故障数
        lines = f.readlines()[::2]  # 故障辞書ファイルのid情報を読み込む
        st_fault_all_line = [int(line.split()[3]) for line in lines[::2]]  # 縮退故障信号線番号を取得
        # print(st_fault_num)
        # print("st_fault_all_line：", st_fault_all_line)
        # print(suplit_num)

    # 故障診断対象の回路における「ブリッジ故障」の総数を取得
    with open(part_brdic_file + str(0), 'r') as f:  # ブリッジ故障辞書ファイルの1つを開く
        fault_inf = f.readline()  # 故障情報
        br_fault_num = int(fault_inf.split()[2]) # 対象回路で起こりうる故障数
        lines = f.readlines()[::2]  # 故障辞書ファイルのid情報を読み込む
        br_fault_all_line = [int(line.split()[3]) for line in lines[::10]]  # ブリッジ故障が発生する可能性がある信号線番号（支配される信号線の番号）をリストに格納　※縮退故障と異なり、出力信号線以外にもブリッジ故障が発生しない信号線があるため、故障辞書から読み込まないといけない
        # br_fault_type = [int(line.split()[4]) for line in lines[::2]] # ブリッジ故障における故障の種類（0、1どちらに支配されるのか）を取得
        # br_dominate_line = [int(line.split()[5]) for line in lines[::2]]  # 支配する信号線を格納
        # print(br_fault_num)
        # print("br_fault_all_line：", br_fault_all_line)
    
    
    
  # 学習済みの機械学習モデルに入力データを与えて、出力を取得する
    # 入力データをファイルから読み込む
    input_data = mk_input_data()

    # 分割モデルの数を取得
    with open(suplit_num_file, 'r') as f:
        num_models = int(f.readline())
    
    # 診断対象の回路における故障信号線の総数を取得
    with open(cir + 'output', 'r') as f:
        line = f.readline().replace(",", "").replace("\n", "")
        global faulet_line_num
        fault_line_num = int(len(line))  # 診断対象の故障信号線の総数
        # print("fault_line_num" + str(fault_line_num))


    # ネットリストを開いて、入力信号線数、出力信号線数、その他の信号線数を取得
    with open('c' + cir, 'r') as f:
        line_inf = f.readline()  # 1行目はネットリストの情報
        output_line_num = int(line_inf.split()[1])
        input_line_num = int(line_inf.split()[2])
    
    # 統合されていない信号線があるかどうかを取得 0: 統合されてない信号線がない、1: 統合されていない信号線がある
    # 統合されていない信号線がある場合、最後のモデルの最後の出力ノードの値は統合されていない
    with open(single_line_file, 'r') as f:
        single_flag = int(f.readline().replace("\n", ""))


    # データを何個づつ分割したのかを取得
    with open(suplit_data_num_file, 'r') as f:
        suplit_data_num = int(f.readline().replace("\n", ""))  # データの分割数を取得
        # print(suplit_data_num)

    for model_id in range(num_models):

        # モデルの読み込み
        with open(model_folder + '/' + cir + 'model_' + str(model_id) + '.tflite', 'rb') as f:  # モデルを読み込むファイルを開く
            tflite_model = f.read()

        # モデルの評価
        interpreter = tf.lite.Interpreter(model_content=tflite_model)  # TFLite形式のモデルを読み込む。保存されたTFLiteモデルをメモリに読み込み、推論を行う準備をするためのインタープリターを作成します。
        interpreter.allocate_tensors()  # #メモリを確保。モデルが使用するテンソル（データ構造）をメモリに割り当てます。TFLiteモデルをインタープリターにロードするだけでは、テンソルのメモリは割り当てられていません。この行を実行することで、モデルが推論に必要なメモリを確保します。

        input_details = interpreter.get_input_details()  # モデルの入力テンソルの詳細を取得。モデルの入力に関する詳細情報をリスト形式で返します。各要素は、入力テンソルの形状、データ型、名前などの情報を含む辞書です。
        output_details = interpreter.get_output_details()  # モデルの出力テンソルの詳細を取得。モデルの出力に関する詳細情報をリスト形式で返します。各要素は、出力テンソルの形状、データ型、名前などの情報を含む辞書です。

        correct_file = correct_data_file + str(model_id)  # 分割された正解データファイル名に変更　＝　s344分割正解データ/s344integrated_output + 番号
        mk_output_data(correct_file)  # 関数内でグローバル変数output_node_numを設定
        

        n = 0
        f = open('check' + cir + 'output_data' + str(model_id), 'w')  # 正解データファイルを開く
        for i in range(input_data_num):
            
            input_shape = input_details[0]['shape']  # 入力データの形状を取得.先ほど取得したinput_detailsのリストの中から、形状情報を取得。input_details[0]['shape']は、入力テンソルの形状を取得するためのコードです。['shape']は、辞書内の shape キーを指定しています。

            reshape_input_data = np.reshape(input_data[i], input_shape)  # NumPy配列の形状を指定した形 (input_shape) に変更します。これにより、入力データの形状がモデルの入力テンソルの形状と一致します。


            # モデルの入力データを設定
            interpreter.set_tensor(input_details[0]['index'], np.array(reshape_input_data, dtype=np.float32))   # モデルの入力テンソルにデータを設定します。入力テンソルのインデックス、データを指定します。[0]['index']は、入力テンソルのインデックスを取得するためのコードです。

            # モデルの推論を実行
            interpreter.invoke()

            # モデルの出力データを取得
            output_data = interpreter.get_tensor(output_details[0]['index'])
            
            if (model_id == (num_models - 1)) and single_flag == 1:  # 最後のモデルで、統合されていない信号線がある場合
                for j in range(output_node_num):
                    if j < (output_node_num - 1):  # 最後の出力ノード以外の場合
                        if output_data[0][j] < 0.02:
                            output_data[0][j] = '0.000'
                        elif output_data[0][j] < 0.45:
                            output_data[0][j] = '0.15'
                        elif output_data[0][j] < 0.98:
                            output_data[0][j] = '0.75'
                        else:
                            output_data[0][j] = '1.000'
                    else:  # 統合されていない信号線がある場合、最後の出力ノードは統合されていない信号線の値を持つため、それは₀または1の値を持つため、0.5を閾値として0または1に変換する
                        if output_data[0][j] <= 0.5:
                            output_data[0][j] = '0.000'
                        else:
                            output_data[0][j] = '1.000'
                    if j < output_node_num - 1:  # 最後の要素以外はカンマを付ける
                      f.write(str(output_data[0][j]) + ',')
                    else:                                  # 最後の要素はカンマを付けない
                      f.write(str(output_data[0][j]) + '\n')
            else:  # 最後のモデルで、統合されていない信号線がない場合、または最後のモデル以外の場合
                for j in range(output_node_num):
                    if output_data[0][j] < 0.02:
                        output_data[0][j] = '0.000'
                    elif output_data[0][j] < 0.45:
                        output_data[0][j] = '0.15'
                    elif output_data[0][j] < 0.98:
                        output_data[0][j] = '0.75'
                    else:
                        output_data[0][j] = '1.000'
                    
                    if j < output_node_num - 1:  # 最後の要素以外はカンマを付ける
                      f.write(str(output_data[0][j]) + ',')
                    else:                                  # 最後の要素はカンマを付けない
                      f.write(str(output_data[0][j]) + '\n')


22
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
output_node_num: 21
